# 🌲 Random Forest & Gradient Boosting — Baseline sur dataset .npy
## Score de référence ML classique sur le même test set que le modèle Conformer
### CMKL University · Stage 2026

---

**Objectif** : établir des baselines ML classiques sur le dataset synthétique officiel
de Dr. Sarun pour comparer directement avec le modèle Conformer PatchTST (95.25%).

**Dataset** : fichiers `.npy` officiels, grille 6700 points, bruit `Upto30SNR` (-30dB)

| Modèle | Type | Features testées |
|--------|------|------------------|
| Random Forest | Ensemble d'arbres | Spectre brut / PCA / Features spectrales |
| XGBoost | Gradient Boosting | idem |
| LightGBM | Gradient Boosting | idem |

**Point de comparaison** :
```
PatchTST Conformer (notre modèle) : 95.25% Test officiel
RF/GB (ce notebook)               : ?
```


---
## ⚙️ Section 0 — Installation & Imports


In [ ]:
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['scikit-learn', 'xgboost', 'lightgbm', 'seaborn']:
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        install(pkg)

print('✓ Packages prêts')

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
import xgboost as xgb
import lightgbm as lgb

SEED = 42
np.random.seed(SEED)
HOME = os.path.expanduser('~')
print(f'HOME = {HOME}')
print('✓ Imports OK')

---
## 📊 Section 1 — Chargement des données .npy

Exactement le même dataset et le même test set que le modèle Conformer PatchTST.
C'est la condition nécessaire pour une comparaison directe et honnête.


In [ ]:
# ── Chemins (à adapter selon ton organisation sur Glider) ────────────────
DATA_ROOT = os.path.join(HOME, 'data', '2026-FTIR-Preprocesed')
TRAIN_DIR = os.path.join(DATA_ROOT, '1.1 TrainingSet - UptoY dB')
TEST_DIR  = os.path.join(DATA_ROOT, '1.2 TestSet - UptoY dB')

# ── Niveau de bruit — DOIT être identique au run Conformer ────────────────
NOISE_VARIANT = 'Upto30SNR'

def npy_path(base_dir, filename):
    p = os.path.join(base_dir, filename)
    if not os.path.exists(p):
        print(f'  ✗ INTROUVABLE : {p}')
    return p

print(f'Chargement niveau de bruit : {NOISE_VARIANT}')
train_clean = np.load(npy_path(TRAIN_DIR, 'TrainGroundTruthSet_Pre.npy'))
train_noisy = np.load(npy_path(TRAIN_DIR, f'TrainNoisySet_{NOISE_VARIANT}_Pre.npy'))
test_clean  = np.load(npy_path(TEST_DIR,  'TestGroundTruthSet_Pre.npy'))
test_noisy  = np.load(npy_path(TEST_DIR,  f'TestNoisySet_{NOISE_VARIANT}_Pre.npy'))

print(f'\ntrain_noisy : {train_noisy.shape}')
print(f'test_noisy  : {test_noisy.shape}')

In [ ]:
# ── Labels (ordre alphabétique confirmé avec Dr. Sarun) ──────────────────
ASSUMED_CLASSES = [
    'ABS', 'ACRYLIC', 'CELLULOSE', 'CHITOSAN', 'ENR', 'EPDM', 'EVA', 'HDPE',
    'LDPE', 'NYLON', 'PBAT', 'PBS', 'PC', 'PEEK', 'PEI', 'PET',
    'PF THERMOPLASTIC', 'PF THERMOSET', 'PHB', 'PLA', 'PMMA', 'POM', 'PP',
    'PS', 'PTFE', 'PU', 'PVA', 'PVC', 'PVDF', 'SAN',
]
N_CLASSES    = len(ASSUMED_CLASSES)
N_PER_CLASS  = train_noisy.shape[0] // N_CLASSES

le = LabelEncoder()
le.fit(ASSUMED_CLASSES)

y_train = np.repeat(np.arange(N_CLASSES), N_PER_CLASS)
y_test  = np.repeat(np.arange(N_CLASSES), test_noisy.shape[0] // N_CLASSES)

print(f'{N_CLASSES} classes × {N_PER_CLASS} spectres train')
print(f'{N_CLASSES} classes × {test_noisy.shape[0]//N_CLASSES} spectres test')

In [ ]:
# ── Instance normalisation (identique au notebook Conformer) ──────────────
def instance_norm(X):
    """Normalise chaque spectre individuellement (μ=0, σ=1)."""
    mu    = X.mean(axis=1, keepdims=True)
    sigma = X.std(axis=1, keepdims=True) + 1e-8
    return (X - mu) / sigma

X_train_raw = instance_norm(train_noisy).astype(np.float32)
X_test_raw  = instance_norm(test_noisy).astype(np.float32)

print(f'X_train_raw : {X_train_raw.shape}')
print(f'X_test_raw  : {X_test_raw.shape}')

---
## 🔧 Section 2 — Extraction de features

3 représentations testées, du plus simple au plus riche :

| Feature set | Dimension | Description |
|-------------|-----------|-------------|
| Spectre brut normalisé | 6700 | Toute l'information, haute dimension |
| PCA 50 composantes | 50 | Résumé statistique (même approche que notebook CSV) |
| Features spectrales FTIR | ~70 | Bandes chimiques clés + ratios |

⚠️ **Sur 6700 dimensions, RF sur spectre brut sera très lent** — PCA sera
probablement le meilleur compromis vitesse/performance.


In [ ]:
# ── Grille spectrale du dataset .npy ──────────────────────────────────────
WN_GRID = np.arange(650, 4000, 0.5)   # 6700 points, pas 0.5 cm⁻¹
assert len(WN_GRID) == X_train_raw.shape[1], \
    f'Grille {len(WN_GRID)} pts != spectres {X_train_raw.shape[1]} pts'

# ── Bandes FTIR adaptées à la résolution 0.5 cm⁻¹ ─────────────────────────
FTIR_BANDS = [
    ('CH3_asym',    2962, 15), ('CH2_asym',    2926, 15), ('CH2_sym',     2853, 15),
    ('CH_bend',     1460, 15), ('CH3_rock',    1375, 12), ('CO_ester',    1735, 20),
    ('CO_carbonat', 1775, 20), ('CO_amide',    1650, 20), ('CO_stretch1', 1260, 20),
    ('CO_stretch2', 1100, 25), ('CO_stretch3', 1020, 20), ('arom_ring',   1600, 15),
    ('arom_CH',     3030, 15), ('arom_oop',     700, 20), ('OH_broad',    3330, 80),
    ('NH_stretch',  3300, 30), ('CCl_stretch',  690, 20), ('CF_stretch',  1210, 30),
    ('CC_alkene',   1640, 15), ('CC_diene',     970, 15),
]

def extract_spectral_features(X):
    """Extrait ~70 features spectrales chimiquement significatives."""
    n = X.shape[0]
    features = []
    eps = 1e-8

    for sp in X:
        feat = {}
        for name, center, hw in FTIR_BANDS:
            mask = (WN_GRID >= center-hw) & (WN_GRID <= center+hw)
            if mask.sum() == 0:
                feat[f'mean_{name}'] = 0.0
                feat[f'peak_{name}'] = 0.0
            else:
                feat[f'mean_{name}'] = float(sp[mask].mean())
                feat[f'peak_{name}'] = float(sp[mask].max())

        feat['ratio_CH2_CH3']    = feat['mean_CH2_asym']  / (feat['mean_CH3_asym']   + eps)
        feat['ratio_CO_CH2']     = feat['mean_CO_ester']  / (feat['mean_CH2_asym']   + eps)
        feat['ratio_OH_CH2']     = feat['mean_OH_broad']  / (feat['mean_CH2_asym']   + eps)
        feat['ratio_CCl_CH2']    = feat['mean_CCl_stretch']/(feat['mean_CH2_asym']   + eps)
        feat['ratio_arom_aliph'] = feat['mean_arom_ring'] / (feat['mean_CH2_asym']   + eps)

        feat['global_mean'] = float(sp.mean())
        feat['global_std']  = float(sp.std())
        feat['global_max']  = float(sp.max())
        feat['global_skew'] = float(pd.Series(sp).skew())

        for lbl, lo, hi in [('fingerprint',650,1500), ('functional',1500,2000),
                              ('CH_region',2700,3000), ('OH_region',3000,3700)]:
            m = (WN_GRID >= lo) & (WN_GRID <= hi)
            if m.sum() > 0:
                feat[f'region_{lbl}_mean'] = float(sp[m].mean())
                feat[f'region_{lbl}_max']  = float(sp[m].max())
                feat[f'region_{lbl}_std']  = float(sp[m].std())
        features.append(feat)

    return pd.DataFrame(features).values.astype(np.float32)


# ── PCA ───────────────────────────────────────────────────────────────────
print('PCA en cours...')
pca = PCA(n_components=50, random_state=SEED)
X_train_pca = pca.fit_transform(X_train_raw)
X_test_pca  = pca.transform(X_test_raw)
print(f'  Variance expliquée : {pca.explained_variance_ratio_.sum():.1%}')

# ── Features spectrales ───────────────────────────────────────────────────
print('Extraction features spectrales (train)...')
X_train_feat = extract_spectral_features(X_train_raw)
print('Extraction features spectrales (test)...')
X_test_feat  = extract_spectral_features(X_test_raw)

# ── Combined : PCA + features spectrales ──────────────────────────────────
X_train_comb = np.hstack([X_train_feat, X_train_pca])
X_test_comb  = np.hstack([X_test_feat,  X_test_pca])

FEATURE_SETS = {
    'PCA 50 comp.'       : (X_train_pca,  X_test_pca),
    'Features spectrales': (X_train_feat, X_test_feat),
    'Combined (feat+PCA)': (X_train_comb, X_test_comb),
}
# Note : spectre brut (6700d) omis par défaut — très lent sur RF
# Décommenter si tu veux le tester :
# FEATURE_SETS['Spectre brut (6700d)'] = (X_train_raw, X_test_raw)

print(f'\n✓ Features prêtes')
for name, (Xtr, Xte) in FEATURE_SETS.items():
    print(f'  {name:25s} : train={Xtr.shape}, test={Xte.shape}')

---
## 🌲 Section 3 — Random Forest


In [ ]:
RF_PARAMS = dict(
    n_estimators = 500,
    max_features = 'sqrt',
    class_weight = 'balanced',
    n_jobs       = -1,
    random_state = SEED,
)

rf_results = {}
print(f'=== Random Forest (500 arbres) ===')
print(f'{"Feature set":25s} | {"Test Acc":>8} | {"Test F1":>8}')
print('-' * 50)

for name, (Xtr, Xte) in FEATURE_SETS.items():
    rf = RandomForestClassifier(**RF_PARAMS)
    rf.fit(Xtr, y_train)
    preds = rf.predict(Xte)
    acc   = accuracy_score(y_test, preds)
    f1    = f1_score(y_test, preds, average='macro', zero_division=0)
    rf_results[name] = {'model': rf, 'acc': acc, 'f1': f1, 'preds': preds}
    print(f'{name:25s} | {acc:8.2%} | {f1:8.3f}')

best_rf_name  = max(rf_results, key=lambda k: rf_results[k]['acc'])
best_rf_acc   = rf_results[best_rf_name]['acc']
best_rf_preds = rf_results[best_rf_name]['preds']
print(f'\n→ Meilleur RF : {best_rf_name} ({best_rf_acc:.2%})')

In [ ]:
print(f'=== Rapport RF ({best_rf_name}) ===')
print(classification_report(y_test, best_rf_preds,
                             target_names=le.classes_, zero_division=0))

---
## ⚡ Section 4 — XGBoost


In [ ]:
XGB_PARAMS = dict(
    n_estimators     = 300,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    eval_metric      = 'mlogloss',
    random_state     = SEED,
    n_jobs           = -1,
)

xgb_results = {}
print(f'=== XGBoost (300 estimateurs) ===')
print(f'{"Feature set":25s} | {"Test Acc":>8} | {"Test F1":>8}')
print('-' * 50)

for name, (Xtr, Xte) in FEATURE_SETS.items():
    clf = xgb.XGBClassifier(**XGB_PARAMS)
    clf.fit(Xtr, y_train, eval_set=[(Xte, y_test)], verbose=False)
    preds = clf.predict(Xte)
    acc   = accuracy_score(y_test, preds)
    f1    = f1_score(y_test, preds, average='macro', zero_division=0)
    xgb_results[name] = {'model': clf, 'acc': acc, 'f1': f1, 'preds': preds}
    print(f'{name:25s} | {acc:8.2%} | {f1:8.3f}')

best_xgb_name  = max(xgb_results, key=lambda k: xgb_results[k]['acc'])
best_xgb_acc   = xgb_results[best_xgb_name]['acc']
best_xgb_preds = xgb_results[best_xgb_name]['preds']
print(f'\n→ Meilleur XGB : {best_xgb_name} ({best_xgb_acc:.2%})')

---
## 🔥 Section 5 — LightGBM


In [ ]:
LGB_PARAMS = dict(
    n_estimators     = 300,
    num_leaves       = 63,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    class_weight     = 'balanced',
    random_state     = SEED,
    n_jobs           = -1,
    verbose          = -1,
)

lgb_results = {}
print(f'=== LightGBM (300 estimateurs) ===')
print(f'{"Feature set":25s} | {"Test Acc":>8} | {"Test F1":>8}')
print('-' * 50)

for name, (Xtr, Xte) in FEATURE_SETS.items():
    clf = lgb.LGBMClassifier(**LGB_PARAMS)
    clf.fit(Xtr, y_train)
    preds = clf.predict(Xte)
    acc   = accuracy_score(y_test, preds)
    f1    = f1_score(y_test, preds, average='macro', zero_division=0)
    lgb_results[name] = {'model': clf, 'acc': acc, 'f1': f1, 'preds': preds}
    print(f'{name:25s} | {acc:8.2%} | {f1:8.3f}')

best_lgb_name  = max(lgb_results, key=lambda k: lgb_results[k]['acc'])
best_lgb_acc   = lgb_results[best_lgb_name]['acc']
best_lgb_preds = lgb_results[best_lgb_name]['preds']
print(f'\n→ Meilleur LGB : {best_lgb_name} ({best_lgb_acc:.2%})')

---
## 📊 Section 6 — Comparaison finale avec PatchTST Conformer

**Même test set officiel .npy, même niveau de bruit Upto30SNR (-30dB)**


In [ ]:
# ── Tableau comparatif ────────────────────────────────────────────────────
rows = []
for name, res in rf_results.items():
    rows.append({'Modèle': f'RF — {name}', 'Test Acc': res['acc'], 'Macro F1': res['f1']})
for name, res in xgb_results.items():
    rows.append({'Modèle': f'XGB — {name}', 'Test Acc': res['acc'], 'Macro F1': res['f1']})
for name, res in lgb_results.items():
    rows.append({'Modèle': f'LGB — {name}', 'Test Acc': res['acc'], 'Macro F1': res['f1']})

# Ajouter le PatchTST Conformer comme référence
PATCHTST_ACC = 0.9525
rows.append({'Modèle': 'PatchTST Conformer ⭐', 'Test Acc': PATCHTST_ACC, 'Macro F1': None})

df_summary = pd.DataFrame(rows).sort_values('Test Acc', ascending=False)
df_summary['Test Acc'] = df_summary['Test Acc'].map('{:.2%}'.format)
df_summary['Macro F1'] = df_summary['Macro F1'].map(lambda x: f'{x:.3f}' if x else 'N/A')
print(df_summary.to_string(index=False))

In [ ]:
# ── Graphique de comparaison ──────────────────────────────────────────────
all_models = {}
for k, v in rf_results.items():  all_models[f'RF\n{k[:12]}']  = v['acc']
for k, v in xgb_results.items(): all_models[f'XGB\n{k[:12]}'] = v['acc']
for k, v in lgb_results.items(): all_models[f'LGB\n{k[:12]}'] = v['acc']
all_models['PatchTST\nConformer'] = PATCHTST_ACC

colors = ['#2E75B6']*3 + ['#70AD47']*3 + ['#ED7D31']*3 + ['#7030A0']

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(all_models.keys(), [v*100 for v in all_models.values()],
              color=colors, edgecolor='white', width=0.6)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9, fontweight='bold')
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title(
    f'ML Classique vs Deep Learning\n'
    f'Test set officiel .npy — Bruit {NOISE_VARIANT} (-30dB cumulé), 30 classes',
    fontweight='bold', fontsize=13
)
ax.set_ylim(0, 110)
ax.axhline(PATCHTST_ACC*100, color='#7030A0', linestyle='--', alpha=0.4, lw=1.5)
ax.grid(axis='y', alpha=0.3)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#2E75B6', label='Random Forest'),
    Patch(color='#70AD47', label='XGBoost'),
    Patch(color='#ED7D31', label='LightGBM'),
    Patch(color='#7030A0', label='PatchTST Conformer'),
])
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(HOME, 'comparison_models_npy.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Matrice de confusion du meilleur modèle ML ─────────────────────────────
all_res = {}
for k, v in rf_results.items():  all_res[('RF',  k)] = v
for k, v in xgb_results.items(): all_res[('XGB', k)] = v
for k, v in lgb_results.items(): all_res[('LGB', k)] = v

best_key = max(all_res, key=lambda k: all_res[k]['acc'])
best_res = all_res[best_key]

print(f'Meilleur modèle ML : {best_key[0]} — {best_key[1]}')
print(f'Test Accuracy      : {best_res["acc"]:.2%}')
print(f'vs PatchTST        : {PATCHTST_ACC:.2%} (Δ = {PATCHTST_ACC - best_res["acc"]:+.2%})')

cm      = confusion_matrix(y_test, best_res['preds'], labels=range(N_CLASSES))
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
for ax, data, fmt, title in zip(
    axes, [cm, cm_norm], ['d', '.2f'],
    ['Matrice de confusion (counts)', 'Matrice de confusion (normalisée)']):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=le.classes_, yticklabels=le.classes_,
                ax=ax, annot_kws={'size': 7})
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
    ax.tick_params(axis='x', rotation=45)
plt.suptitle(
    f'Meilleur ML : {best_key[0]} ({best_key[1]}) | Acc : {best_res["acc"]:.2%}',
    fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Résumé final ──────────────────────────────────────────────────────────
print('═'*65)
print(f'  RÉSUMÉ — Dataset .npy, {NOISE_VARIANT} (-30dB), 30 classes')
print('═'*65)
print(f'  Random Forest (best)     : {best_rf_acc:.2%}')
print(f'  XGBoost (best)           : {best_xgb_acc:.2%}')
print(f'  LightGBM (best)          : {best_lgb_acc:.2%}')
print(f'  PatchTST Conformer       : {PATCHTST_ACC:.2%}  (architecture multi-tâche)')
print(f'  Meilleur ML global       : {max(best_rf_acc, best_xgb_acc, best_lgb_acc):.2%}')
print(f'  Écart PatchTST vs ML     : {PATCHTST_ACC - max(best_rf_acc, best_xgb_acc, best_lgb_acc):+.2%}')
print('═'*65)